# Introduction to Transfer Learning with EfficientNet

In this notebook, we dive into the concept of transfer learning using EfficientNet, a state-of-the-art convolutional neural network (CNN) architecture known for its efficiency in model scaling. Transfer learning allows us to leverage pre-trained models on large datasets like ImageNet and fine-tune them for specific tasks, such as classifying images in the CIFAR-10 dataset.

We'll start by building a simple CNN from scratch to establish a baseline. Then, we'll explore EfficientNet's architecture, focusing on its compound scaling method that balances network depth, width, and resolution to achieve high performance with fewer parameters and FLOPs compared to previous models. We'll implement EfficientNet variants for CIFAR-10, train them from scratch, and compare their performance. Finally, we'll apply transfer learning using a pre-trained EfficientNet model from torchvision, demonstrating both feature extraction and fine-tuning approaches.

This hands-on tutorial is designed for beginners, assuming no prior knowledge of deep learning. We'll explain each step in detail, including code, visualizations, and comparisons. By the end, you'll understand how EfficientNet improves efficiency and how transfer learning can boost accuracy with limited data.

## Step 1: Historical Background of CNNs Before EfficientNet

Convolutional Neural Networks (CNNs) have evolved dramatically since their early days. Understanding this progression helps explain why EfficientNet was such an important breakthrough in 2019.

### Early Foundations (1980s–2000s)
- 1989: LeNet-5 by Yann LeCun — the first practical CNN for handwritten digit recognition
- 1998–2012: CNNs remained relatively shallow (5–10 layers) due to vanishing gradients, limited compute, and small datasets

### Deep Learning Revival (2012–2015)
- 2012: AlexNet (8 layers) wins ImageNet → deep CNNs become feasible with ReLU, dropout, data augmentation, and GPUs
- 2014: VGG (16–19 layers) → showed that **very deep but simple** stacked 3×3 convolutions work surprisingly well
- 2015: The major breakthrough — **ResNet** (He et al., 2015) introduced residual (skip) connections, enabling training of networks with 50, 101, even 152 layers

### Post-ResNet Challenges (2016–2018)
After ResNet solved the degradation problem for very deep networks, researchers tried to push performance further by:
- Simply making networks deeper (e.g. ResNet-1001)
- Wider (more channels/filters)
- Higher resolution input

However, these naive scaling approaches led to **rapidly increasing computational cost** (FLOPs) with only marginal accuracy gains.

### Need for EfficientNet (2019)
Most scaling attempts before 2019 treated depth, width, and resolution independently and arbitrarily.  
This resulted in **inefficient** models — large compute budgets but suboptimal accuracy.

**EfficientNet** (Tan & Le, 2019) introduced a principled, data-driven solution:

> **Compound scaling**: uniformly scale depth, width, **and** resolution using fixed coefficients (α, β, γ) discovered via small-scale grid search.

This compound scaling rule produces a family of models (EfficientNet-B0 to B7) that achieve **state-of-the-art accuracy with dramatically fewer parameters and FLOPs** than previous models (including ResNet, DenseNet, NASNet, AmoebaNet, etc.).

Key paper:  
Tan, M., & Le, Q. V. (2019). **EfficientNet: Rethinking Model Scaling for Convolutional Neural Networks**. International Conference on Machine Learning (ICML).

Especially read:
- Section 3.1: Compound Model Scaling
- Section 4: Experiments & Scaling Coefficients
- Table 1: Comparison of different scaling dimensions

In this notebook we will:
- See the limitations of naive scaling ourselves (plain deep networks)
- Build small EfficientNet-like models from scratch using MBConv blocks
- Compare them to plain (non-residual) networks
- Finally apply transfer learning with the official pre-trained EfficientNet-B0 from torchvision

**References used in this section:**

- LeCun, Y., et al. (1989). Backpropagation Applied to Handwritten Zip Code Recognition. Neural Computation.
- Krizhevsky, A., Sutskever, I., & Hinton, G. E. (2012). ImageNet Classification with Deep Convolutional Neural Networks. NeurIPS.
- Simonyan, K., & Zisserman, A. (2014). Very Deep Convolutional Networks for Large-Scale Image Recognition. arXiv:1409.1556.
- He, K., Zhang, X., Ren, S., & Sun, J. (2015). Deep Residual Learning for Image Recognition. CVPR.
- **Tan, M., & Le, Q. V. (2019). EfficientNet: Rethinking Model Scaling for Convolutional Neural Networks. ICML.**

# Notebook Objectives
By the end of this notebook, you will:

1. Understand the historical context leading to EfficientNet's development, including challenges with inefficient model scaling.
2. Learn about the degradation problem in deep networks and how EfficientNet's compound scaling addresses efficiency.
3. Implement basic building blocks: Plain convolutional blocks vs. EfficientNet's MBConv (Mobile Inverted Bottleneck Convolution) blocks with inverted residuals and squeeze-excitation.
4. Build small EfficientNet-like architectures adapted for CIFAR-10 (e.g., EfficientNet-B0 inspired with scaled-down stages).
5. Train networks from scratch and compare plain vs. EfficientNet variants to see efficiency gains in accuracy vs. parameters/FLOPs.
6. Explore transfer learning: Use pre-trained EfficientNet-B0 from torchvision on ImageNet, adapt it to CIFAR-10.
7. Compare training from scratch vs. feature extraction vs. fine-tuning.
8. Visualize training curves, compare histories, and interpret results.

We'll use PyTorch for implementation, focusing on beginner-friendly explanations with references to the EfficientNet paper (Tan & Le, 2019):

- Section 3.1 for compound scaling method (balancing α-depth, β-width, γ-resolution).
- Section 4 for experiments showing SOTA efficiency (e.g., EfficientNet-B7 reaches 84.3% ImageNet top-1 with 66M params vs. ResNet-152's 77.8% with 60M params).
- Table 1 for baselines on different scaling dimensions.

# Why start with CIFAR-10?
CIFAR-10 is a classic benchmark dataset for image classification, consisting of 60,000 32x32 color images in 10 classes (e.g., airplane, automobile, bird, cat, deer, dog, frog, horse, ship, truck). It's small enough to train on modest hardware but challenging enough to demonstrate the benefits of advanced architectures like EfficientNet.
Why CIFAR-10 for this tutorial?

- Beginner-friendly: Low resolution (32x32) means faster training than high-res datasets like ImageNet.
- Reveals architecture strengths: Simple datasets highlight how EfficientNet's efficient scaling (via MBConv blocks and compound coefficients) improves accuracy with fewer resources compared to plain networks.
- Transfer learning demo: Pre-trained EfficientNet models are on ImageNet (224x224), so we can show adaptation to smaller images via resizing/interpolation.
- Comparisons: We'll run experiments like:
    - Simple CNN from scratch (baseline)
    - EfficientNet-B0 from scratch
    - Pre-trained EfficientNet-B0 + Transfer Learning (feature extraction vs. fine-tuning)


This setup mirrors real-world scenarios where you start with small data and leverage pre-trained efficient models for resource-constrained environments.

Reference: Tan & Le (2019), Section 4.3: CIFAR experiments showing EfficientNet's efficiency on small datasets.

---

## Imports & Basic Setup

First, we import the necessary libraries and set up the device (GPU if available).

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models  # For pre-trained EfficientNet
from torchsummary import summary  # For model summaries

import matplotlib.pyplot as plt
import numpy as np
from typing import List, Dict

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

**Explanation:**
- `torch` and `nn`: Core PyTorch for models and layers.
- `optim`: Optimizers like Adam.
- `torchvision`: Datasets (CIFAR-10), transforms, and pre-trained models including EfficientNet variants.
- `torchsummary`: To print model architecture summaries.
- `matplotlib` and `numpy`: For visualizations.
- Device setup: Use GPU for faster training if available.

Reference: PyTorch documentation for torchvision.models.efficientnet (supports EfficientNet-B0 to B7 pre-trained on ImageNet).

## Hyperparameters

Define the key hyperparameters for training. These can be tuned later for better performance.

In [ ]:
# Hyperparameters
BATCH_SIZE = 128
LEARNING_RATE = 0.001  # We'll adjust for different experiments
NUM_EPOCHS_SCRATCH = 30  # For from-scratch training
NUM_EPOCHS_TRANSFER = 15  # Shorter for transfer learning
MOMENTUM = 0.9
WEIGHT_DECAY = 1e-4

**Explanation:**
- `BATCH_SIZE`: Number of samples per gradient update. Larger is faster but uses more memory.
- `LEARNING_RATE`: Step size for optimizer. We'll use different values for scratch vs. transfer.
- `NUM_EPOCHS_*`: Training iterations over the dataset.
- `MOMENTUM` and `WEIGHT_DECAY`: For SGD optimizer (if used); helps with convergence.

For EfficientNet experiments, we'll primarily use Adam optimizer with lr=1e-4 for stability, as recommended in the paper for smaller models (Tan & Le, 2019, Section 4.1: Training details).

## Data transforms (preprocessing + augmentation)
To prepare the CIFAR-10 data, we define transforms for preprocessing and data augmentation. This improves generalization and handles EfficientNet's input requirements (e.g., normalization to ImageNet stats).

In [ ]:
# Data transforms for training (with augmentation)
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),  # Random crop with padding
    transforms.RandomHorizontalFlip(),     # Random horizontal flip
    transforms.ToTensor(),                 # Convert to tensor
    transforms.Normalize(                  # Normalize with ImageNet stats (EfficientNet pre-trained expectation)
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ),
])

# Data transforms for validation (no augmentation)
transform_val = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ),
])

Explanation:

- Augmentation (train only): RandomCrop and HorizontalFlip add variety, helping prevent overfitting.
- ToTensor: Converts PIL images to PyTorch tensors.
- Normalize: Uses ImageNet mean/std since we'll use pre-trained EfficientNet later. For scratch training, this still helps convergence.
- Why these values? They match the statistics used for ImageNet training, as per EfficientNet paper (Tan & Le, 2019, Section 4.1: Data preprocessing).

Note: CIFAR-10 is 32x32, but EfficientNet expects 224x224 for pre-trained models. We'll handle resizing in transfer learning sections.

## Download and prepare CIFAR-10
Now, download the CIFAR-10 dataset and create data loaders for training and validation.

In [ ]:
# Download CIFAR-10 dataset
trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_train)
valset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_val)

# Data loaders
trainloader = torch.utils.data.DataLoader(trainset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
valloader = torch.utils.data.DataLoader(valset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

# Class labels for visualization
classes = ('airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck')

Explanation:

-   `datasets.CIFAR10` : Loads the dataset, downloading if not present.
-   `root='./data'`: Storage directory.
-   `transform`: Applies the predefined transforms.
-   `DataLoader`: Batches the data, shuffles training set for better generalization.
-   `num_workers=2`: Uses 2 subprocesses for faster data loading.

This setup is efficient for small datasets like CIFAR-10, aligning with EfficientNet's focus on resource efficiency (Tan & Le, 2019, Section 4.3: Experiments on CIFAR).

## Visualize some training images
Let's visualize a batch of training images to get a feel for the dataset. We'll display 25 images with their labels.
Python

In [ ]:
def imshow(img):
    # Denormalize for visualization
    img = img * torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1) + torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    npimg = img.numpy()
    plt.imshow(np.transpose(npimg, (1, 2, 0)))
    plt.show()

# Get a batch of training data
dataiter = iter(trainloader)
images, labels = next(dataiter)

# Make a grid from 25 images
img_grid = torchvision.utils.make_grid(images[:25], nrow=5)

# Show images and labels
imshow(img_grid)
print(' '.join(f'{classes[labels[j]]:8s}' for j in range(25)))

Explanation:

-   `imshow`: Helper function to denormalize and display the image grid using matplotlib.
-   `Denormalization`: Reverses the Normalize transform to show original colors.
-   `make_grid`: Arranges images into a 5x5 grid.
-   Prints the class labels below the grid.

This helps verify data loading and transforms. Note: Augmentation means each run shows different variations.
Reference: Standard PyTorch tutorial practice for CIFAR-10 visualization.

## Core training & validation functions

Here we define two core functions:
- `train_epoch`: Trains the model for one epoch on the training set.
- `validate`: Evaluates the model on the validation set.

These are modular and can be used in our training loop.

In [ ]:
def train_epoch(model: nn.Module, dataloader: torch.utils.data.DataLoader, criterion: nn.Module, 
                optimizer: optim.Optimizer, device: torch.device) -> float:
    model.train()  # Set model to training mode
    running_loss = 0.0
    correct = 0
    total = 0
    
    for batch_idx, (inputs, labels) in enumerate(dataloader):
        inputs, labels = inputs.to(device), labels.to(device)
        
        optimizer.zero_grad()  # Zero the parameter gradients
        
        outputs = model(inputs)  # Forward pass
        loss = criterion(outputs, labels)
        
        loss.backward()  # Backward pass
        optimizer.step()  # Optimize
        
        running_loss += loss.item()
        
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
    
    epoch_loss = running_loss / len(dataloader)
    epoch_acc = correct / total
    return epoch_loss, epoch_acc

def validate(model: nn.Module, dataloader: torch.utils.data.DataLoader, criterion: nn.Module, 
             device: torch.device) -> float:
    model.eval()  # Set model to evaluation mode
    running_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs, labels = inputs.to(device), labels.to(device)
            
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item()
            
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
    
    val_loss = running_loss / len(dataloader)
    val_acc = correct / total
    return val_loss, val_acc

**Explanation:**
- `train_epoch`: Loops over batches, computes forward/backward, updates weights. Tracks loss and accuracy.
- `validate`: Similar but no gradients/updates; uses `torch.no_grad()` for efficiency.
- Both return average loss and accuracy for the epoch.
- These functions are general and work for any classification model, including our EfficientNet variants.

This setup is efficient for tracking progress, aligning with EfficientNet's emphasis on compute efficiency (Tan & Le, 2019, Section 4).

## Complete training loop + visualization (train_and_evaluate)

Now, we combine the core functions into a full training loop that:
- Trains for multiple epochs
- Tracks history (loss/acc for train/val)
- Prints progress
- Returns the history for later comparisons

We'll also define a plotting function to visualize the curves.

In [ ]:
def train_and_evaluate(model: nn.Module, trainloader: torch.utils.data.DataLoader, 
                       valloader: torch.utils.data.DataLoader, criterion: nn.Module, 
                       optimizer: optim.Optimizer, num_epochs: int, device: torch.device) -> Dict[str, List[float]]:
    history = {
        'train_loss': [], 'train_acc': [],
        'val_loss': [], 'val_acc': []
    }
    
    for epoch in range(num_epochs):
        train_loss, train_acc = train_epoch(model, trainloader, criterion, optimizer, device)
        val_loss, val_acc = validate(model, valloader, criterion, device)
        
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        
        print(f'Epoch {epoch+1}/{num_epochs}: '
              f'Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}, '
              f'Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}')
    
    return history

def plot_training_curves(history: Dict[str, List[float]], title: str = 'Training Curves'):
    epochs = range(1, len(history['train_loss']) + 1)
    
    plt.figure(figsize=(12, 4))
    
    # Plot loss
    plt.subplot(1, 2, 1)
    plt.plot(epochs, history['train_loss'], label='Train Loss')
    plt.plot(epochs, history['val_loss'], label='Val Loss')
    plt.title(f'{title} - Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    
    # Plot accuracy
    plt.subplot(1, 2, 2)
    plt.plot(epochs, history['train_acc'], label='Train Acc')
    plt.plot(epochs, history['val_acc'], label='Val Acc')
    plt.title(f'{title} - Accuracy')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.legend()
    
    plt.tight_layout()
    plt.show()

**Explanation:**
- `train_and_evaluate`: Runs the loop, calls train_epoch/validate, collects history in a dict.
- Prints per-epoch stats for monitoring.
- `plot_training_curves`: Uses matplotlib to plot loss and acc curves side-by-side.
- This is reusable for all our experiments, allowing easy comparison of EfficientNet vs. plain networks.

Reference: Standard practice in PyTorch tutorials; aligns with monitoring training in EfficientNet paper (Tan & Le, 2019, Section 4.1: Used similar loops with early stopping, but we keep it simple here).

## Comparison Function for Multiple Training Histories

To make fair comparisons between different models (plain vs. EfficientNet, scratch vs. transfer, etc.), we define a function that plots multiple training histories side-by-side.

In [ ]:
def compare_histories(histories: Dict[str, Dict[str, List[float]]], title: str = 'Model Comparison'):
    """
    Plots training/validation loss and accuracy curves for multiple models.
    
    Args:
        histories: dict where key = model name, value = history dict from train_and_evaluate
        title: Plot title
    """
    plt.figure(figsize=(14, 5))
    
    # Loss plot
    plt.subplot(1, 2, 1)
    for name, hist in histories.items():
        epochs = range(1, len(hist['val_loss']) + 1)
        plt.plot(epochs, hist['val_loss'], label=f'{name} Val Loss')
    plt.title(f'{title} - Validation Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    # Accuracy plot
    plt.subplot(1, 2, 2)
    for name, hist in histories.items():
        epochs = range(1, len(hist['val_acc']) + 1)
        plt.plot(epochs, hist['val_acc'], label=f'{name} Val Acc')
    plt.title(f'{title} - Validation Accuracy')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

**How to use it (example later):**

```python
# After training several models
histories = {
    "Plain-20": history_plain20,
    "EfficientNet-small": history_eff_small,
    # ...
}
compare_histories(histories, title="Plain vs EfficientNet")

This function helps us visually see which architecture learns faster, generalizes better, or suffers from degradation — exactly what we want when comparing naive deep networks vs. EfficientNet-style designs.

## The Degradation Problem & Why We Need EfficientNet

One of the key motivations behind modern efficient architectures like EfficientNet is solving (or rather avoiding) the **degradation problem** observed in very deep plain networks — and then going beyond by scaling more intelligently than simply stacking more layers.

### The Degradation Problem (observed in plain networks)
When we make a very deep network using only plain convolutional blocks (no skip connections), accuracy often **saturates quickly and then degrades** as depth increases — even on training data.

This is **not** caused by overfitting (since train error also increases), but rather by optimization difficulties: gradients vanish/explode, or the network struggles to learn identity mappings through many layers.

ResNet (2015) solved this for depth by introducing **skip connections** → allowing the network to learn residuals instead of direct mappings.

But even after ResNet, simply making networks deeper, wider, or higher-resolution independently still led to **very poor efficiency**: huge increases in FLOPs for only small accuracy gains.

### EfficientNet's Answer: Compound Scaling + MBConv Blocks

EfficientNet (Tan & Le, 2019) revisited scaling with a principled, empirical approach:

1. They ran a small-scale grid search on depth (d), width (w), and resolution (r) scaling factors.
2. Found that the **optimal balance** follows a simple compound rule:

   depth = α^φ  
   width = β^φ  
   resolution = γ^φ  

   where φ is a compound coefficient that controls the overall compute budget, and α ≈ 1.2, β ≈ 1.1, γ ≈ 1.15 were discovered to be near-optimal.

   → This uniform scaling across **all three dimensions** gives dramatically better accuracy per FLOP than scaling only one dimension.

3. They also used a more efficient building block: **MBConv** (Mobile Inverted Bottleneck Convolution) with **squeeze-and-excitation (SE)** attention.

   - Inverted residual: expand channels → depthwise conv → project back (like MobileNetV2)
   - SE module: channel-wise attention to focus on important features
   - Swish activation instead of ReLU

This combination made each block **much more parameter- and FLOP-efficient** than standard residual blocks.

### Visualizing the MBConv Block

Here is a simplified diagram of the MBConv block used in EfficientNet:


Input → 1×1 expansion conv (×6) → depthwise 3×3/5×5 conv → SE module → 1×1 projection conv → Add skip (if channels match & stride=1) → Output


(Reference: Tan & Le, 2019, Figure 4 – MBConv block with SE)

## Plain BasicBlock (without efficient features)

To demonstrate the degradation problem ourselves, we first implement a **plain convolutional block** — similar to the basic building block used in very early deep networks (e.g., pre-ResNet VGG-style stacked convs without any skip connections).

In [ ]:
class PlainBlock(nn.Module):
    """
    Plain convolutional block (no skip connection)
    Used to show degradation when we stack many layers deeply.
    """
    def __init__(self, in_channels, out_channels, stride=1):
        super(PlainBlock, self).__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1   = nn.BatchNorm2d(out_channels)
        self.relu  = nn.ReLU(inplace=True)
        
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(out_channels)
        
        self.stride = stride
        # No skip connection here → this is what causes degradation in deep plain nets

    def forward(self, x):
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out = self.relu(out)
        return out

**Key characteristics of PlainBlock:**
- Two 3×3 convolutions
- BatchNorm + ReLU after each conv
- **No residual / skip connection**
- When we stack many of these (e.g., 20+ layers), optimization becomes very hard → accuracy drops even on training set

This reproduces the exact degradation problem described in He et al. (2015) before residual connections were introduced.

In contrast, EfficientNet uses **MBConv blocks** (inverted residuals + SE), which are both deeper per block and much more efficient.

Next, we'll implement the MBConv block used in EfficientNet.

## Residual BasicBlock equivalent for EfficientNet (MBConv block)

EfficientNet does **not** use the classic ResNet-style residual block (two 3×3 convs with identity skip).  
Instead, it uses the **MBConv** (Mobile Inverted Bottleneck Convolution) block, introduced in MobileNetV2 and significantly enhanced in EfficientNet with:

- Inverted residual structure (expand → depthwise → project)
- Squeeze-and-Excitation (SE) attention module
- Swish activation (better than ReLU in most cases)
- Optional dropout and stochastic depth (not implemented here for simplicity)

Below is a clean PyTorch implementation of the MBConv block used in EfficientNet.

In [ ]:
import torch.nn.functional as F

class SEModule(nn.Module):
    """Squeeze-and-Excitation module (channel attention)"""
    def __init__(self, channels, reduction=4):
        super(SEModule, self).__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.fc1 = nn.Conv2d(channels, channels // reduction, kernel_size=1, bias=True)
        self.relu = nn.ReLU(inplace=True)
        self.fc2 = nn.Conv2d(channels // reduction, channels, kernel_size=1, bias=True)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        module_input = x
        x = self.avg_pool(x)
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        x = self.sigmoid(x)
        return module_input * x


class MBConvBlock(nn.Module):
    """
    Mobile Inverted Bottleneck Convolution block with Squeeze-Excitation.
    This is the core building block of EfficientNet.
    """
    def __init__(self, in_channels, out_channels, expansion_ratio=6, kernel_size=3,
                 stride=1, se_ratio=0.25, drop_rate=0.0):
        super(MBConvBlock, self).__init__()
        
        self.stride = stride
        self.has_se = se_ratio > 0
        self.drop_rate = drop_rate
        
        # Expansion phase: 1×1 conv to increase channels
        hidden_dim = int(in_channels * expansion_ratio)
        if expansion_ratio != 1:
            self.expand_conv = nn.Conv2d(in_channels, hidden_dim, 1, bias=False)
            self.bn0 = nn.BatchNorm2d(hidden_dim)
        else:
            self.expand_conv = None
        
        # Depthwise convolution
        self.depthwise_conv = nn.Conv2d(
            hidden_dim, hidden_dim, kernel_size, stride, kernel_size//2, groups=hidden_dim, bias=False
        )
        self.bn1 = nn.BatchNorm2d(hidden_dim)
        
        # Squeeze-Excitation
        if self.has_se:
            self.se = SEModule(hidden_dim, int(hidden_dim * se_ratio))
        
        # Projection phase: 1×1 conv to reduce channels
        self.project_conv = nn.Conv2d(hidden_dim, out_channels, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        
        # Activation (EfficientNet uses Swish)
        self.swish = lambda x: x * torch.sigmoid(x)
        
        # Skip connection only if input/output shape matches
        self.use_residual = in_channels == out_channels and stride == 1

    def forward(self, x):
        identity = x
        
        # Expansion
        if self.expand_conv is not None:
            x = self.swish(self.bn0(self.expand_conv(x)))
        else:
            x = x
        
        # Depthwise
        x = self.swish(self.bn1(self.depthwise_conv(x)))
        
        # SE
        if self.has_se:
            x = self.se(x)
        
        # Projection
        x = self.bn2(self.project_conv(x))
        
        # Residual connection + optional dropout
        if self.use_residual:
            if self.drop_rate > 0:
                x = F.dropout2d(x, p=self.drop_rate, training=self.training)
            x = x + identity
        
        return x

**Key differences from classic ResNet BasicBlock:**
- Inverted bottleneck: expands channels first (×6 usually), then depthwise conv, then compresses back
- Depthwise separable conv → far fewer parameters/FLOPs
- SE module: learns which channels are important
- Swish activation: smoother gradients than ReLU
- No 3×3 convs in parallel — uses depthwise + pointwise

This block is **much more efficient** per layer, allowing EfficientNet to go deeper and wider within the same compute budget.

Reference: Tan & Le (2019), Figure 4 and Section 3.2 – MBConv with SE is the main building block.

## Full EfficientNet Class Adapted for CIFAR-10 (3 Stages – similar scaling)

Now we define a simplified but faithful version of an EfficientNet-style network adapted for CIFAR-10 (32×32 images).

Key adaptations for CIFAR-10:
- Smaller initial stem (3×3 conv instead of 3×3 stride-2 + 3×3 stride-2)
- Fewer and smaller stages (only 3 main stages instead of 7)
- Reduced channel counts and depths compared to full EfficientNet-B0
- Final classifier adapted to 10 classes

This lets us clearly see the benefit of MBConv + compound-like scaling even on small images.

In [ ]:
class EfficientNetCIFAR(nn.Module):
    def __init__(self, block=MBConvBlock, depths=[1, 2, 4], channels=[16, 40, 112],
                 expansion_ratios=[1, 6, 6], kernel_sizes=[3, 5, 3], se_ratios=[0.25, 0.25, 0.25]):
        super().__init__()
        
        # Stem
        self.stem = nn.Sequential(
            nn.Conv2d(3, channels[0], kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(channels[0]),
            nn.SiLU()
        )
        
        self.stages = nn.ModuleList()
        in_ch = channels[0]
        
        is_plain = block is PlainBlock   # or check block.__name__ == 'PlainBlock'
        
        for i in range(len(depths)):
            stage = nn.Sequential()
            for j in range(depths[i]):
                stride = 2 if j == 0 and i > 0 else 1
                
                if is_plain:
                    # only pass supported arguments
                    block_instance = block(
                        in_channels=in_ch,
                        out_channels=channels[i],
                        stride=stride if j == 0 else 1
                    )
                else:
                    # full MBConv arguments
                    block_instance = block(
                        in_channels=in_ch,
                        out_channels=channels[i],
                        expansion_ratio=expansion_ratios[i],
                        kernel_size=kernel_sizes[i],
                        stride=stride if j == 0 else 1,
                        se_ratio=se_ratios[i]
                    )
                
                stage.add_module(f'block_{j}', block_instance)
                in_ch = channels[i]
            
            self.stages.append(stage)
        
        # Head
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(in_ch, 10)
        )
    def forward(self, x):
        x = self.stem(x)
        
        for stage in self.stages:
            x = stage(x)
        
        x = self.head(x)
        return x

**Notes on this implementation:**
- Stem is very light (single 3×3 stride-1 conv) — suitable for 32×32 input
- 3 stages only → roughly mimics EfficientNet-B0's early-to-mid stages
- Downsampling happens only at the start of stage 2 and 3 (stride=2)
- Uses `nn.SiLU()` (official Swish in PyTorch ≥1.7)
- No dropout/stochastic depth here to keep it simple for teaching

In real EfficientNet-B0: 7 stages, 224×224 input, many more blocks/channels — but the principle is the same.

## Convenient Factory Function: efficientnet_cifar10()

To make it easy to create different variants (similar to how EfficientNet has B0–B7 with compound scaling), we define a factory function.

This lets us quickly instantiate models with different "scaling levels" by adjusting depths, channels, etc.

In [ ]:
def efficientnet_cifar10(variant='B0', block=MBConvBlock):
    """
    Factory function to create EfficientNet-like models for CIFAR-10.
    
    Variants loosely inspired by EfficientNet scaling:
    - 'plain20' / 'plain32': use PlainBlock (for degradation demo)
    - 'small'  : very light (≈ EfficientNet-B0 level for small images)
    - 'medium' : moderately deeper/wider
    """
    if variant.startswith('plain'):
        block = PlainBlock  # override to use plain blocks
        if variant == 'plain20':
            depths = [3, 3, 3, 3]          # total ~20 conv layers
            channels = [16, 32, 64, 128]
        elif variant == 'plain32':
            depths = [5, 5, 5, 5]          # deeper → should show degradation
            channels = [16, 32, 64, 128]
        else:
            raise ValueError("Unknown plain variant")
        
        return EfficientNetCIFAR(
            block=block,
            depths=depths,
            channels=channels,
            expansion_ratios=[1]*len(depths),   # plain has no expansion
            kernel_sizes=[3]*len(depths),
            se_ratios=[0]*len(depths)           # no SE in plain
        )
    
    elif variant == 'small':
        # Roughly B0-inspired for CIFAR
        return EfficientNetCIFAR(
            block=block,
            depths=[1, 2, 4],               # total blocks ~7
            channels=[16, 40, 112],
            expansion_ratios=[1, 6, 6],
            kernel_sizes=[3, 5, 3],
            se_ratios=[0.25, 0.25, 0.25]
        )
    
    elif variant == 'medium':
        # Slightly scaled up (like moving toward B1/B2)
        return EfficientNetCIFAR(
            block=block,
            depths=[2, 3, 5, 2],            # more blocks
            channels=[24, 48, 136, 192],
            expansion_ratios=[1, 6, 6, 6],
            kernel_sizes=[3, 5, 3, 5],
            se_ratios=[0.25]*4
        )
    
    else:
        raise ValueError(f"Unknown variant: {variant}")

**Usage examples (coming in next cells):**
- `efficientnet_cifar10('plain20')` → plain deep net (to show degradation)
- `efficientnet_cifar10('plain32')` → even deeper plain net
- `efficientnet_cifar10('small')`   → EfficientNet-style small model
- `efficientnet_cifar10('medium')`  → scaled-up variant

This mirrors how the official EfficientNet family scales via compound coefficient φ.

Reference: Tan & Le (2019), Section 3.1 & Table 2 – compound scaling produces family of models.

## Create plain20 = efficientnet_cifar10('plain20') + summary + training

Now we create our first model: a **plain20** network using only PlainBlock (no residuals, no SE, no expansion — classic stacked convs).

This should demonstrate degradation when we go deeper later.

In [ ]:
# Instantiate plain20 model
plain20 = efficientnet_cifar10(variant='plain20').to(device)

# Print model summary (input size 3×32×32 for CIFAR-10)
print("Plain20 Model Summary:")
summary(plain20, (3, 32, 32))

**Expected output (approximate):**
- Total params: ~0.5–1M (depending on exact channel counts)
- Much fewer parameters than full EfficientNet, but stacking plain blocks deeply will hurt training.

In [ ]:
# Train plain20 from scratch
print("\nTraining Plain20 (30 epochs)...")
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(plain20.parameters(), lr=1e-3, weight_decay=1e-4)

history_plain20 = train_and_evaluate(
    model=plain20,
    trainloader=trainloader,
    valloader=valloader,
    criterion=criterion,
    optimizer=optimizer,
    num_epochs=NUM_EPOCHS_SCRATCH,  # 30
    device=device
)

# Plot its training curves
plot_training_curves(history_plain20, title="Plain20 Training Curves")

**Observations to note after training:**
- Training may start reasonably but often plateaus or even degrades in deeper plain nets.
- Validation accuracy typically stays low (~60–75% range after 30 epochs on CIFAR-10 with this setup).
- This is exactly the behavior that motivated residual connections in 2015 — and efficient blocks later.

## Create and train plain32 (deeper variant)

To observe the **degradation problem** more clearly, we now create a deeper plain network (`plain32`) using the same `PlainBlock` but with more layers per stage.

This should show that — without residual connections — simply going deeper hurts performance rather than helping.

In [ ]:
# Instantiate plain32 (deeper plain network)
plain32 = efficientnet_cifar10(variant='plain32').to(device)

# Model summary
print("Plain32 Model Summary:")
summary(plain32, (3, 32, 32))

**Comparison note:**
- plain32 has roughly 50–70% more layers/parameters than plain20
- In theory (with good optimization), deeper should be better — but in practice with plain blocks, it usually performs **worse** or barely improves.

In [ ]:
# Train plain32 (same setup: Adam, lr=1e-3, 30 epochs)
print("\nTraining Plain32 (30 epochs)...")
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(plain32.parameters(), lr=1e-3, weight_decay=1e-4)

history_plain32 = train_and_evaluate(
    model=plain32,
    trainloader=trainloader,
    valloader=valloader,
    criterion=criterion,
    optimizer=optimizer,
    num_epochs=NUM_EPOCHS_SCRATCH,
    device=device
)

# Plot its curves
plot_training_curves(history_plain32, title="Plain32 Training Curves")

**What to look for:**
- Compare final validation accuracy of plain32 vs plain20
- Check whether deeper plain network has **higher training loss** or **lower accuracy** — this is the degradation phenomenon.
- In most runs, plain32 will underperform plain20 despite having more capacity.

## Compare plain20 vs plain32 histories

Now that both plain models have been trained, let's directly compare their validation performance side-by-side.

This visual comparison clearly shows the **degradation problem**: deeper plain networks (without residuals or efficient blocks) usually do **not** improve — and often perform worse — despite having more capacity.

In [ ]:
# Collect histories
plain_histories = {
    "Plain20": history_plain20,
    "Plain32": history_plain32
}

# Compare validation curves
compare_histories(plain_histories, title="Plain20 vs Plain32 (Deeper Plain Network)")

**Typical observations after running:**
- Plain20 usually reaches higher validation accuracy than Plain32.
- Plain32 may show **higher validation loss** and/or slower convergence (or even degradation after some epochs).
- Training curves often reveal that the deeper plain model struggles more with optimization.

This is the exact issue that residual connections (ResNet) solved in 2015 — and that EfficientNet further improves upon with MBConv + compound scaling.

## Create and train small EfficientNet-style model (MBConv-based)

Now we switch to an **EfficientNet-inspired small model** using proper MBConv blocks (inverted residuals + SE + Swish).

This should significantly outperform the plain networks — even with fewer or similar parameters — thanks to more efficient building blocks.

In [ ]:
# Instantiate the small EfficientNet variant
eff_small = efficientnet_cifar10(variant='small').to(device)

# Model summary
print("EfficientNet-Small (CIFAR-adapted) Model Summary:")
summary(eff_small, (3, 32, 32))

**Quick comparison (approximate numbers):**
- Plain20/Plain32: ~0.5–1M params, classic convs
- eff_small: usually similar or slightly more params, but **much more expressive per parameter** due to MBConv + SE

Expect noticeably better validation accuracy than plain models.

In [ ]:
# Train the small EfficientNet model
print("\nTraining EfficientNet-Small (30 epochs)...")
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(eff_small.parameters(), lr=1e-3, weight_decay=1e-4)

history_eff_small = train_and_evaluate(
    model=eff_small,
    trainloader=trainloader,
    valloader=valloader,
    criterion=criterion,
    optimizer=optimizer,
    num_epochs=NUM_EPOCHS_SCRATCH,
    device=device
)

# Plot curves
plot_training_curves(history_eff_small, title="EfficientNet-Small Training Curves")

**What to expect:**
- Faster convergence than plain models
- Higher final validation accuracy (often 80–88%+ on CIFAR-10 after 30 epochs with this setup)
- Smoother loss curves thanks to better gradient flow and SE attention

## Compare efficientnet small vs plain small

We now directly compare the **EfficientNet-style small model** (with MBConv + SE) against the **plain20** baseline.

This highlights the core efficiency advantage of EfficientNet's building blocks — even at similar parameter counts / compute budgets, MBConv-based models usually learn **faster** and reach **higher accuracy**.

In [ ]:
# Collect the relevant histories
small_comparison_histories = {
    "Plain20 (no residuals)": history_plain20,
    "EfficientNet-Small (MBConv + SE)": history_eff_small
}

# Side-by-side comparison of validation performance
compare_histories(
    small_comparison_histories,
    title="Plain20 vs EfficientNet-Small (CIFAR-10)"
)

**Typical observations from this plot:**
- **EfficientNet-Small** almost always reaches higher validation accuracy (often +5–15% over plain20 after 30 epochs).
- Its validation loss decreases more steadily and continues improving longer.
- The gap widens over epochs — showing better optimization and representation power per parameter.
- This mirrors what Tan & Le (2019) demonstrated: MBConv + squeeze-excitation + Swish + careful scaling beats naive stacking of plain or even classic residual blocks.

Next, we will scale up a bit further — create and train a **deeper / wider EfficientNet-style variant** ("medium") and see if compound-like scaling continues to pay off.

## Compare efficientnet small vs deeper

We now compare the **small EfficientNet variant** against the **medium/deeper variant** we trained earlier.

This shows whether applying a mild form of compound scaling (more blocks + wider channels) continues to bring gains — even on a small dataset like CIFAR-10.

In [ ]:
# Assuming you have already trained the medium variant earlier
# If not, run the training cell for 'medium' first (similar to eff_small)

eff_deeper = efficientnet_cifar10(variant='medium').to(device)

# (Re-)train if needed — skip if you already have history_eff_medium
print("\nTraining EfficientNet-Medium / deeper variant (30 epochs)...")
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(eff_deeper.parameters(), lr=1e-3, weight_decay=1e-4)

history_eff_medium = train_and_evaluate(
    model=eff_deeper,
    trainloader=trainloader,
    valloader=valloader,
    criterion=criterion,
    optimizer=optimizer,
    num_epochs=NUM_EPOCHS_SCRATCH,
    device=device
)

plot_training_curves(history_eff_medium, title="EfficientNet-Medium (Deeper) Training Curves")

In [ ]:
# Compare small vs deeper EfficientNet-style models
deeper_comparison_histories = {
    "EfficientNet-Small": history_eff_small,
    "EfficientNet-Medium (deeper/wider)": history_eff_medium
}

compare_histories(
    deeper_comparison_histories,
    title="EfficientNet-Small vs EfficientNet-Medium (Scaling Effect)"
)

**What this comparison typically reveals:**
- The **medium** variant (more blocks, slightly wider channels) often reaches **higher final accuracy** than the small one.
- Gains are usually modest but consistent (e.g., +2–6% val acc), showing that careful scaling still helps.
- Training curves may show that the deeper model converges a bit slower initially but pulls ahead later — classic sign of increased capacity being used effectively (unlike plain nets).
- This is the essence of EfficientNet's contribution: **balanced scaling across depth/width/resolution** gives better returns than arbitrary increases in one dimension.

## Compare efficientnet deeper vs plain deeper

Finally, let's compare the **deeper EfficientNet-style model** (medium variant with MBConv + SE) against the **deeper plain network** (plain32).

This should show a **huge gap**: while deepening hurts plain nets (degradation), it helps EfficientNet-style nets thanks to efficient blocks and balanced scaling.

In [ ]:
# Collect histories for this comparison
deeper_vs_plain_histories = {
    "Plain32 (deeper plain)": history_plain32,
    "EfficientNet-Medium (deeper MBConv)": history_eff_medium
}

# Side-by-side validation comparison
compare_histories(
    deeper_vs_plain_histories,
    title="Deeper Plain vs Deeper EfficientNet-Style (MBConv Advantage)"
)

**Key insights from this final from-scratch comparison:**
- **EfficientNet-Medium** typically achieves much higher validation accuracy than Plain32 (often +10–20% or more).
- Plain32 may plateau early or even degrade, while EfficientNet-Medium keeps improving.
- Parameter efficiency: EfficientNet variants often reach better results with similar or fewer params/FLOPs due to MBConv's design (depthwise convs + SE).
- This demonstrates why EfficientNet (Tan & Le, 2019) set new SOTA on efficiency: compound scaling + advanced blocks beat naive deep plain or even classic residual nets.

We've now seen the benefits of EfficientNet's architecture from scratch. Next: **Transfer Learning** — leveraging a pre-trained EfficientNet-B0 from torchvision on ImageNet for even better results on CIFAR-10 with less training.

---

# Transfer Learning section

We've now seen how EfficientNet-style architectures (with MBConv blocks and compound scaling) outperform plain networks when trained from scratch on CIFAR-10.
But training from scratch requires lots of data and compute. Enter Transfer Learning: leverage a model pre-trained on a large dataset (e.g., ImageNet-1K with 1.28M images) and adapt it to our smaller task.
For EfficientNet, torchvision provides pre-trained weights for EfficientNet-B0 (and higher variants) on ImageNet.

We'll demonstrate:

- Training EfficientNet-B0 from scratch (random init) on CIFAR-10 → baseline
- Feature Extraction: Freeze pre-trained backbone, only train new classifier
- Fine-Tuning: Unfreeze some/all layers, train with small LR

Since ImageNet uses 224×224 inputs, we'll resize CIFAR-10 images to 224×224 in transforms.
Reference: Tan & Le (2019), Section 4.2: Transfer learning experiments show EfficientNet-B0 pre-trained achieves high accuracy on downstream tasks with fine-tuning.
## Update Transforms for 224×224 Input
EfficientNet expects larger inputs, so we add resizing.

In [ ]:
# Updated transforms for transfer learning (resize to 224x224)
transform_train_tl = transforms.Compose([
    transforms.Resize(224),                # Resize to EfficientNet input size
    transforms.RandomCrop(224, padding=28),# Random crop with padding
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

transform_val_tl = transforms.Compose([
    transforms.Resize(224),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# Reload datasets with new transforms
trainset_tl = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_train_tl)
valset_tl = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_val_tl)

# Data loaders (smaller batch size for larger images)
BATCH_SIZE_TL = 64  # Adjust if memory issues
trainloader_tl = torch.utils.data.DataLoader(trainset_tl, batch_size=BATCH_SIZE_TL, shuffle=True, num_workers=2)
valloader_tl = torch.utils.data.DataLoader(valset_tl, batch_size=BATCH_SIZE_TL, shuffle=False, num_workers=2)

Explanation:

Resize to 224×224 matches pre-trained EfficientNet input.
Larger padding in RandomCrop for more augmentation.
Reload datasets/loaders — now with bigger images, so smaller batch size to fit in memory.

## Load Pre-trained EfficientNet-B0 and Adapt to CIFAR-10

In [ ]:
def get_efficientnet_b0(pretrained=True, num_classes=10):
    model = models.efficientnet_b0(pretrained=pretrained)
    # Adapt classifier for CIFAR-10 (ImageNet has 1000 classes)
    model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)
    return model

Notes:

- models.efficientnet_b0(pretrained=True): Loads ImageNet weights.
- Replace final linear layer: in_features=1280 → 10 outputs.
- For scratch: set pretrained=False.

## Train EfficientNet-B0 from Scratch
First, baseline: train EfficientNet-B0 with random initialization.


In [ ]:
# EfficientNet-B0 from scratch
eff_b0_scratch = get_efficientnet_b0(pretrained=False).to(device)

# Summary
summary(eff_b0_scratch, (3, 224, 224))

# Train (15 epochs, Adam lr=1e-3)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(eff_b0_scratch.parameters(), lr=1e-3, weight_decay=1e-4)

history_eff_b0_scratch = train_and_evaluate(
    eff_b0_scratch, trainloader_tl, valloader_tl, criterion, optimizer, NUM_EPOCHS_TRANSFER, device
)

plot_training_curves(history_eff_b0_scratch, title="EfficientNet-B0 Scratch")

Expectations: Good but not great (~80–85% val acc) — full EfficientNet-B0 is large (5M params), but scratch on 50K images limits it.

## Feature Extraction: Freeze Backbone, Train Classifier Only

In [ ]:
# Pre-trained EfficientNet-B0 for feature extraction
eff_b0_feat_extract = get_efficientnet_b0(pretrained=True).to(device)

# Freeze all parameters except classifier
for param in eff_b0_feat_extract.parameters():
    param.requires_grad = False
for param in eff_b0_feat_extract.classifier.parameters():
    param.requires_grad = True

# Summary (note frozen params)
summary(eff_b0_feat_extract, (3, 224, 224))

# Train (15 epochs, higher lr for classifier)
optimizer = optim.Adam(eff_b0_feat_extract.classifier.parameters(), lr=1e-3, weight_decay=1e-4)

history_eff_b0_feat_extract = train_and_evaluate(
    eff_b0_feat_extract, trainloader_tl, valloader_tl, criterion, optimizer, NUM_EPOCHS_TRANSFER, device
)

plot_training_curves(history_eff_b0_feat_extract, title="EfficientNet-B0 Feature Extraction")

: 

Benefits: Fast training (only ~10K params updated), good for small data. Expect ~85–90% val acc.

## Fine-Tuning: Unfreeze All, Small LR

In [ ]:
# Pre-trained EfficientNet-B0 for fine-tuning
eff_b0_finetune = get_efficientnet_b0(pretrained=True).to(device)

# Unfreeze all (but use small LR to avoid destroying pre-trained weights)
optimizer = optim.Adam(eff_b0_finetune.parameters(), lr=1e-4, weight_decay=1e-4)  # Smaller LR

history_eff_b0_finetune = train_and_evaluate(
    eff_b0_finetune, trainloader_tl, valloader_tl, criterion, optimizer, NUM_EPOCHS_TRANSFER, device
)

plot_training_curves(history_eff_b0_finetune, title="EfficientNet-B0 Fine-Tuning")

Expectations: Best results (~90–95%+ val acc) — pre-trained features + full adaptation.

## Compare All Transfer Learning Approaches

In [ ]:
tl_histories = {
    "Scratch": history_eff_b0_scratch,
    "Feature Extract": history_eff_b0_feat_extract,
    "Fine-Tune": history_eff_b0_finetune
}

compare_histories(tl_histories, title="EfficientNet-B0: Scratch vs Feature Extract vs Fine-Tune")

Final Insights:

- Transfer learning >> scratch, especially fine-tuning.
- Aligns with EfficientNet paper: pre-trained models transfer well due to efficient scaling.

This concludes the notebook! You've now mastered transfer learning with EfficientNet on CIFAR-10.